# Getting Started with `bw_timex`: explicit processes and products

This notebook is a product-explicit variant of [`getting_started.ipynb`](./getting_started.ipynb). It keeps the same tiny A/B/CO2 system, but models the foreground with separate **product** and **process** nodes:

- `A product` is the thing we demand.
- `A process` is the operation that produces `A product` and consumes/emits other things.

Brightway supports both this explicit process/product paradigm and the more common chimaera process+product paradigm. See the Brightway inventory docs on [processes, products, and something in between](https://docs.brightway.dev/en/latest/content/overview/inventory.html#processes-products-and-something-in-between) for the broader data-model discussion. Here we choose the explicit option because it makes output-side timing and production-version-specific foreground evolution easy to inspect. In fleet and stock models, a production or process version date is often called a **vintage**; this notebook uses the plainer phrase first and then treats vintage as that same idea.

<div style="display: flex; justify-content: center; background-color: white; border-radius: 15px; padding: 10px; width: 600px; margin: auto;">
  <img src="data/workflow.svg" style="border-radius: 15px; width: 100%;">
</div>

## The model

The static system is still deliberately simple:

```mermaid
flowchart LR
subgraph background[<i>background</i>]
    B("Process B"):::bg
end

subgraph foreground[<i>foreground</i>]
    AP(["A product"]):::product
    A("A process"):::fg
end

subgraph biosphere[<i>biosphere</i>]
    CO2("CO2"):::bio
end

A-->|"1 unit"|AP
B-->|"3 kg"|A
A-.->|"5 kg"|CO2
B-.->|"11 kg"|CO2

classDef fg color:#222832, fill:#3fb1c5, stroke:none;
classDef product color:#222832, fill:#9c5ffd, stroke:none;
classDef bg color:#222832, fill:#3fb1c5, stroke:none;
classDef bio color:#222832, fill:#9c5ffd, stroke:none;
style background fill:none, stroke:none;
style foreground fill:none, stroke:none;
style biosphere fill:none, stroke:none;
```

The important change is that the foreground process and its reference product are no longer the same node. The functional unit will demand `A product`, while all exchanges live on `A process`.

## Why split products and processes?

In the classic Brightway style, an activity often bundles two concepts: the **process** and its **reference product**. That is compact, but it makes it harder to say whether timing belongs to the product being supplied or to the inputs used by the process.

In the explicit paradigm:

- A **product node** is a thing: `A product`, one kg of material, one vehicle lifetime, one kWh. It is what you demand.
- A **process node** is an operation: `A process`, the recipe that produces the product and consumes other products.
- A **production edge** links the process to the product.

This gives a product a clean upstream and downstream interpretation. The same product can be consumed at one time and produced at another time, and `bw_timex` can carry both timestamps through the graph. This is not the only valid modelling style: chimaera nodes are still widely used and often pragmatic, especially when your source data already has one process tightly coupled to one reference product. The explicit style is useful here because the production-edge RTD can live directly on `A process -> A product` instead of on a structural wrapper activity.

## Project setup and background databases

We create one biosphere flow, one 2020 background database, and one 2030 background database. `B production` emits less CO2 in 2030 than in 2020, so relinking over time has a visible effect.

In [1]:
from datetime import datetime

import bw2data as bd
import numpy as np
from bw_temporalis import TemporalDistribution

bd.projects.set_current("getting_started_explicit_process_product")

# Start from a clean project state so the notebook is reproducible.
for db in list(bd.databases):
    del bd.databases[db]
for method in list(bd.methods):
    del bd.methods[method]

bd.Database("biosphere").write(
    {
        ("biosphere", "CO2"): {
            "type": "emission",
            "name": "CO2",
            "unit": "kg",
        },
    }
)

for database_name, co2_amount in [("background_2020", 11), ("background_2030", 7)]:
    bd.Database(database_name).write(
        {
            (database_name, "B"): {
                "name": "B production",
                "reference product": "B",
                "location": "somewhere",
                "unit": "kg",
                "exchanges": [
                    {
                        "amount": 1,
                        "type": "production",
                        "input": (database_name, "B"),
                    },
                    {
                        "amount": co2_amount,
                        "type": "biosphere",
                        "input": ("biosphere", "CO2"),
                    },
                ],
            },
        }
    )

bd.Method(("our", "method")).write([(("biosphere", "CO2"), 1)])

/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.11/site-packages/bw2calc/__init__.py:54: UserWarning: 
It seems like you have an ARM architecture, but haven't installed scikit-umfpack:

    https://pypi.org/project/scikit-umfpack/

Installing it could give you much faster calculations.

  warnings.warn(UMFPACK_WARNING)
100%|██████████| 1/1 [00:00<00:00, 10837.99it/s]

10:30:45+0200 [info     ] Vacuuming database            
10:30:45+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.



100%|██████████| 1/1 [00:00<00:00, 19599.55it/s]

10:30:45+0200 [info     ] Vacuuming database            
10:30:46+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.



100%|██████████| 1/1 [00:00<00:00, 17848.10it/s]

10:30:46+0200 [info     ] Vacuuming database            


## Relative temporal distributions (RTDs) and foreground evolution

A `TemporalDistribution` with `timedelta64` dates is a relative temporal distribution: its dates are offsets from the time of the consuming exchange.

In this notebook we use three RTDs:

1. `td_a_output` on the **output side** of `A process`: the production edge from `A process` to `A product`.
2. `td_b_input` on the **input side** of `A process`: the exchange where `A process` consumes `B production`.
3. `td_a_to_co2` on the biosphere exchange from `A process` to CO2.

We also add **foreground temporal evolution factors** to the `B production -> A process` exchange. These factors model that later versions of `A process` need less `B` input per unit of `A product`. Here, "version" means the date when the foreground process instance is created by the output-side RTD. In other modelling contexts, especially fleet and stock models, this version date is often called the **vintage**. With `temporal_evolution_reference="consumer"`, the factor is evaluated at that A-process version date (`date_consumer`), not at the background production date (`date_producer`).

In [2]:
td_a_output = TemporalDistribution(
    date=np.array([0, 6], dtype="timedelta64[Y]"),
    amount=np.array([0.6, 0.4]),
)

td_b_input = TemporalDistribution(
    date=np.array([-2, 0, 4], dtype="timedelta64[Y]"),
    amount=np.array([0.3, 0.5, 0.2]),
)

td_a_to_co2 = TemporalDistribution(
    date=np.array([0, 1], dtype="timedelta64[Y]"),
    amount=np.array([0.6, 0.4]),
)

process_version_efficiency_factors = {
    datetime(2024, 1, 1): 1.0,
    datetime(2030, 1, 1): 0.5,
}

### What does an RTD on the output side mean?

The output-side RTD lives on the production edge:

```text
A process --td_a_output--> A product
```

It says when the process supplies the product relative to the demand for that product. With a demand in 2024 and `td_a_output = [0 years: 60%, +6 years: 40%]`, `bw_timex` creates two time-specific instances of `A process`:

- 60% of the demanded product is produced in 2024.
- 40% is produced in 2030.

This is useful for production-time grouping or delivery timing. In a fleet model, the product could be a vehicle lifetime, and the output-side RTD would place vehicle production across production years.

### What does an RTD on the input side mean?

The input-side RTD lives on an exchange consumed by the process:

```text
B production --td_b_input--> A process
```

It says when the input is needed relative to each time-specific instance of `A process`. With `td_b_input = [-2 years: 30%, 0 years: 50%, +4 years: 20%]`, every `A process` instance consumes its `B` input partly two years before, partly in the same year, and partly four years later.

### What happens when both sides have RTDs?

They are combined. The output-side RTD first splits the process into production dates. Then each input-side RTD is applied relative to each of those dates.

For this notebook:

- `A process` in 2024 consumes `B` in 2022, 2024, and 2028.
- `A process` in 2030 consumes `B` in 2028, 2030, and 2034.

The timeline therefore carries two dates for input rows:

- `date_consumer`: when the consuming foreground process instance exists, here the A-process version date.
- `date_producer`: when the input/background production actually occurs.

### Where do foreground evolution factors fit?

The `process_version_efficiency_factors` are attached to the `B production -> A process` exchange. They scale the foreground exchange amount over time. The confusing part is which time should be used to pick the factor, because the timeline has two relevant dates:

- `date_consumer`: the date of the consuming foreground process instance. In this notebook, this is the A-process version date: 2024 or 2030.
- `date_producer`: the date of the input event. In this notebook, this is when `B production` happens: 2022, 2024, 2028, 2030, or 2034.

Use `temporal_evolution_reference="consumer"` when the change is a property of the consuming process/product version. Here, all inputs belonging to the 2024 A-process version use factor `1.0`, while all inputs belonging to the 2030 A-process version use factor `0.5`. The 2030 A-process version keeps that factor even for an input event in 2034.

Use `temporal_evolution_reference="producer"` when the change is a property of the calendar year in which the exchange event happens. Then the 2034 B input would use the factor for 2034, regardless of whether it belongs to the 2024 or 2030 A-process version.

Rule of thumb:

```text
Is the change a property of the foreground process/product version date?
-> use consumer

Is the change a property of the calendar year when the exchange happens?
-> use producer
```

## Foreground system: explicit product and process

The product node has no exchanges. The process node has all exchanges, including one production edge pointing to the product.

In [3]:
bd.Database("foreground").write(
    {
        ("foreground", "A_product"): {
            "name": "A product",
            "type": bd.labels.product_node_default,
            "unit": "unit",
            "location": "somewhere",
            "exchanges": [],
        },
        ("foreground", "A_process"): {
            "name": "A process",
            "type": bd.labels.process_node_default,
            "unit": "unit",
            "location": "somewhere",
            "exchanges": [
                {
                    "amount": 1,
                    "type": bd.labels.production_edge_default,
                    "input": ("foreground", "A_product"),
                    "temporal_distribution": td_a_output,
                },
                {
                    "amount": 3,
                    "type": bd.labels.consumption_edge_default,
                    "input": ("background_2020", "B"),
                    "temporal_distribution": td_b_input,
                    "temporal_evolution_factors": process_version_efficiency_factors,
                    "temporal_evolution_reference": "consumer",
                },
                {
                    "amount": 5,
                    "type": "biosphere",
                    "input": ("biosphere", "CO2"),
                    "temporal_distribution": td_a_to_co2,
                },
            ],
        },
    }
)

for db in bd.databases:
    bd.Database(db).process()

10:30:46+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 2/2 [00:00<00:00, 4473.92it/s]

10:30:46+0200 [info     ] Vacuuming database            


## Tell `bw_timex` which databases represent which dates

The foreground database is dynamic, because its exchanges are distributed over time. The background databases represent static snapshots at 2020 and 2030.

In [4]:
database_dates = {
    "background_2020": datetime.strptime("2020", "%Y"),
    "background_2030": datetime.strptime("2030", "%Y"),
    "foreground": "dynamic",
}

## Build the timeline

The functional unit demands the product, not the process. This is the main user-facing difference from the classic getting-started notebook.

In [5]:
from bw_timex import TimexLCA

a_product = bd.get_node(database="foreground", code="A_product")

tlca = TimexLCA(
    demand={a_product: 1},
    method=("our", "method"),
    database_dates=database_dates,
)

tlca.build_timeline(starting_datetime=datetime(2024, 1, 1), temporal_grouping="year")

2026-05-04 10:30:46.471 | INFO     | bw_timex.timex_lca:__init__:122 - Initializing TimexLCA object...
2026-05-04 10:30:46.471 | INFO     | bw_timex.timex_lca:__init__:139 - Calculating base LCA...
2026-05-04 10:30:46.482 | INFO     | bw_timex.timex_lca:__init__:156 - Collecting node infos...
2026-05-04 10:30:46.485 | INFO     | bw_timex.timex_lca:build_timeline:252 - No edge filter function provided. Skipping all edges in background databases.
2026-05-04 10:30:46.485 | INFO     | bw_timex.timex_lca:build_timeline:268 - Creating activity time mapping...
2026-05-04 10:30:46.486 | INFO     | bw_timex.timeline_builder:__init__:100 - Traversing supply chain graph...
2026-05-04 10:30:46.491 | INFO     | bw_timex.timeline_builder:build_timeline:156 - Building timeline...
2026-05-04 10:30:46.508 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:563 - Reference date 2034-01-01 00:00:00 is higher than all provided dates. Data will be taken from the close

Starting graph traversal
Calculation count: 1


,date_producer,producer_name,date_consumer,consumer_name,amount,temporal_market_shares
0,2022-01-01,B production,2024-01-01,A process,0.9,"{'background_2020': 0.8, 'background_2030': 0.2}"
1,2024-01-01,B production,2024-01-01,A process,1.5,"{'background_2020': 0.6, 'background_2030': 0.4}"
2,2024-01-01,A process,2024-01-01,-1,0.6,None
3,2028-01-01,B production,2024-01-01,A process,0.6,"{'background_2020': 0.2, 'background_2030': 0.8}"
4,2028-01-01,B production,2030-01-01,A process,0.9,"{'background_2020': 0.2, 'background_2030': 0.8}"
5,2030-01-01,B production,2030-01-01,A process,1.5,{'background_2030': 1}
6,2030-01-01,A process,2030-01-01,-1,0.4,None
7,2034-01-01,B production,2030-01-01,A process,0.6,{'background_2030': 1}


The timeline below shows the combined timing:

- Rows where the producer is `A process` are the output-side RTD: `A product` is supplied in 2024 and 2030.
- Rows where the producer is `B production` are the input-side RTD applied under each A-process date.
- `date_consumer` is the A-process version date. This is the date used by our `consumer`-referenced foreground evolution factors.
- `temporal_market_shares` shows how `bw_timex` relinks each background input to the 2020 and 2030 background snapshots.

In [6]:
tlca.timeline[
    [
        "producer_name",
        "consumer_name",
        "date_producer",
        "date_consumer",
        "amount",
        "temporal_market_shares",
    ]
].sort_values(["date_consumer", "date_producer", "producer_name"])


,producer_name,consumer_name,date_producer,date_consumer,amount,temporal_market_shares
0,B production,A process,2022-01-01,2024-01-01,0.9,"{'background_2020': 0.8, 'background_2030': 0.2}"
2,A process,-1,2024-01-01,2024-01-01,0.6,None
1,B production,A process,2024-01-01,2024-01-01,1.5,"{'background_2020': 0.6, 'background_2030': 0.4}"
3,B production,A process,2028-01-01,2024-01-01,0.6,"{'background_2020': 0.2, 'background_2030': 0.8}"
4,B production,A process,2028-01-01,2030-01-01,0.9,"{'background_2020': 0.2, 'background_2030': 0.8}"
6,A process,-1,2030-01-01,2030-01-01,0.4,None
5,B production,A process,2030-01-01,2030-01-01,1.5,{'background_2030': 1}
7,B production,A process,2034-01-01,2030-01-01,0.6,{'background_2030': 1}


A quick check of the expected foreground timing and the selected process-version factors:

In [8]:
from bw_timex.utils import get_temporal_evolution_factor

b_rows = tlca.timeline[tlca.timeline["producer_name"] == "B production"].copy()
b_rows["A process version year"] = b_rows["date_consumer"].dt.year
b_rows["B production year"] = b_rows["date_producer"].dt.year
b_rows["process-version factor"] = b_rows["date_consumer"].apply(
    lambda date: get_temporal_evolution_factor(process_version_efficiency_factors, date)
)
b_rows["effective B amount"] = b_rows["amount"].astype(float) * b_rows["process-version factor"]

b_rows[
    [
        "A process version year",
        "B production year",
        "amount",
        "process-version factor",
        "effective B amount",
        "temporal_market_shares",
    ]
].sort_values(["A process version year", "B production year"])

,A process version year,B production year,amount,process-version factor,effective B amount,temporal_market_shares
0,2024,2022,0.9,1.0,0.90,"{'background_2020': 0.8, 'background_2030': 0.2}"
1,2024,2024,1.5,1.0,1.50,"{'background_2020': 0.6, 'background_2030': 0.4}"
3,2024,2028,0.6,1.0,0.60,"{'background_2020': 0.2, 'background_2030': 0.8}"
4,2030,2028,0.9,0.5,0.45,"{'background_2020': 0.2, 'background_2030': 0.8}"
5,2030,2030,1.5,0.5,0.75,{'background_2030': 1}
7,2030,2034,0.6,0.5,0.30,{'background_2030': 1}


The unscaled amounts are the edge amount (`3 kg B`) multiplied by the input RTD weights. For each A-process instance, this gives `0.9`, `1.5`, and `0.6` kg of B before foreground evolution.

The process-version-specific factor is then selected from `date_consumer`:

- the 2024 A-process version keeps factor `1.0` for all its B inputs, including the B input produced in 2028;
- the 2030 A-process version keeps factor `0.5` for all its B inputs, including the B input produced in 2034.

That is what version-specific, or vintage-specific, efficiency means here: the efficiency belongs to the foreground process version date (`date_consumer`), not to the calendar year in which each background input happens (`date_producer`).

Aggregating by A-process version year makes the version-specific, or vintage-specific, efficiency difference even clearer:

In [9]:
b_rows.groupby("A process version year", as_index=False).agg(
    raw_B_amount=("amount", "sum"),
    process_version_factor=("process-version factor", "first"),
    effective_B_amount=("effective B amount", "sum"),
)

,A process version year,raw_B_amount,process_version_factor,effective_B_amount
0,2024,3.0,1.0,3.0
1,2030,3.0,0.5,1.5


## Calculate LCI and static LCIA

From here onward, the workflow is the same as in the normal getting-started example. The score includes both background relinking over time and the process-version-specific foreground efficiency factors.

In [10]:
tlca.lci()
tlca.static_lcia()

tlca.static_score

2026-05-04 10:30:46.562 | INFO     | bw_timex.timex_lca:lci:385 - Expanding matrices...
2026-05-04 10:30:46.569 | INFO     | bw_timex.timex_lca:lci:404 - Calculating dynamic inventory...


15.860000000000001

For comparison, the static base LCA uses the original static system without time-explicit relinking or process-version-specific foreground scaling. The time-explicit score is lower because some `B production` is sourced from the cleaner 2030 background database and because the 2030 A-process version uses less B input.

In [11]:
print(f"Static base LCA score:       {tlca.base_lca.score:.2f} kg CO2-eq")
print(f"Time-explicit static score: {tlca.static_score:.2f} kg CO2-eq")

Static base LCA score:       38.00 kg CO2-eq
Time-explicit static score: 15.86 kg CO2-eq


To isolate the foreground evolution effect, we can temporarily remove the process-version factors from the B input exchange and rerun the same time-explicit model. The only difference between the two runs below is the process-version-specific foreground efficiency.

In [12]:
a_process = bd.get_node(database="foreground", code="A_process")
b_input_edge = next(edge for edge in a_process.technosphere() if edge.input["code"] == "B")

saved_factors = b_input_edge["temporal_evolution_factors"]
saved_reference = b_input_edge["temporal_evolution_reference"]
del b_input_edge["temporal_evolution_factors"]
del b_input_edge["temporal_evolution_reference"]
b_input_edge.save()
bd.Database("foreground").process()

tlca_no_evolution = TimexLCA(
    demand={a_product: 1},
    method=("our", "method"),
    database_dates=database_dates,
)
tlca_no_evolution.build_timeline(
    starting_datetime=datetime(2024, 1, 1),
    temporal_grouping="year",
)
tlca_no_evolution.lci()
tlca_no_evolution.static_lcia()

# Restore the foreground exchange so rerunning later cells keeps the intended model.
b_input_edge["temporal_evolution_factors"] = saved_factors
b_input_edge["temporal_evolution_reference"] = saved_reference
b_input_edge.save()
bd.Database("foreground").process()

print(f"Time-explicit score without foreground evolution: {tlca_no_evolution.static_score:.2f} kg CO2-eq")
print(f"Time-explicit score with version-specific efficiencies:   {tlca.static_score:.2f} kg CO2-eq")
print(f"Version-efficiency reduction:                    {tlca_no_evolution.static_score - tlca.static_score:.2f} kg CO2-eq")

2026-05-04 10:30:46.606 | INFO     | bw_timex.timex_lca:__init__:122 - Initializing TimexLCA object...
2026-05-04 10:30:46.606 | INFO     | bw_timex.timex_lca:__init__:139 - Calculating base LCA...
2026-05-04 10:30:46.613 | INFO     | bw_timex.timex_lca:__init__:156 - Collecting node infos...
2026-05-04 10:30:46.615 | INFO     | bw_timex.timex_lca:build_timeline:252 - No edge filter function provided. Skipping all edges in background databases.
2026-05-04 10:30:46.616 | INFO     | bw_timex.timex_lca:build_timeline:268 - Creating activity time mapping...
2026-05-04 10:30:46.616 | INFO     | bw_timex.timeline_builder:__init__:100 - Traversing supply chain graph...
2026-05-04 10:30:46.621 | INFO     | bw_timex.timeline_builder:build_timeline:156 - Building timeline...
2026-05-04 10:30:46.628 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:563 - Reference date 2034-01-01 00:00:00 is higher than all provided dates. Data will be taken from the close

Starting graph traversal
Calculation count: 1
Time-explicit score without foreground evolution: 26.72 kg CO2-eq
Time-explicit score with version-specific efficiencies:   15.86 kg CO2-eq
Version-efficiency reduction:                    10.86 kg CO2-eq


## Optional: dynamic characterization

The inventory still contains dates, so dynamic characterization works in the same way as in the original getting-started notebook.

In [13]:
from dynamic_characterization.ipcc_ar6 import characterize_co2

emission_id = bd.get_activity(("biosphere", "CO2")).id
characterization_functions = {emission_id: characterize_co2}

tlca.dynamic_lcia(
    metric="GWP",
    time_horizon=100,
    characterization_functions=characterization_functions,
)

tlca.dynamic_score

np.float64(15.86)

## Quick recap

The explicit product/process version of the minimal `bw_timex` workflow is:

1. Create a product node for the demanded thing.
2. Create a process node for the operation.
3. Put the output-side RTD on the production edge from process to product when the product supply itself is distributed over time.
4. Put input-side RTDs on the process exchanges when inputs occur before, during, or after each process instance.
5. Add `temporal_evolution_factors` to foreground exchanges when their amounts change by process version date or production year.
6. Use `temporal_evolution_reference="consumer"` for version-specific, or vintage-specific, effects: the factor is evaluated at the process version date (`date_consumer`).
7. Demand the product node in `TimexLCA`.

```python
tlca = TimexLCA(
    demand={a_product: 1},
    method=("our", "method"),
    database_dates=database_dates,
)
tlca.build_timeline(starting_datetime=datetime(2024, 1, 1))
tlca.lci()
tlca.static_lcia()
tlca.static_score
```

The key interpretation is: an output-side RTD controls **when the process instance exists to supply the product**, while an input-side RTD controls **when each input is produced or consumed relative to that process instance**. With `temporal_evolution_reference="consumer"`, foreground evolution factors are selected by the process version date, so later product/process versions can keep their own efficiencies across all later input events.